In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')

In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer

# 지도 학습(supervised learning)과 비지도(unsupervised learning) 학습

머신러닝의 학습 방법은 지도 학습과 비지도 학습으로 나눌 수 있다.

지도 학습이란 라벨링된 데이터를 학습시키는 것을 의미하며, 비지도 학습은 라벨링이 되지 않은 데이터를 학습시키는 것이다. 이 때, 라벨리이란 학습 데이터에 정답이 표시된 것을 의미하며, 정답 부분을 레이블(타겟, 클래스, 실제값) 데이터라고 한다.  

레이블의 형태에 따라 지도 학습은 세부적으로 두 가지 종류로 나눌 수 있다. 레이블이 범주형인 경우 분류(classification) 문제라고 하고, 연속형 숫자인 경우에는 회귀(regression) 문제라고 한다. 머신러닝 알고리즘을 적용하기 전에 레이블의 형태를 파악하고 풀려는 문제가 분류 문제인지 회귀 문제인지 파악하는 것이 중요하다.

# k최근접 이웃(k-Nearest Neighbor, kNN) 알고리즘

k최근접 이웃 알고리즘은 이해하기 쉽고, 자주 사용되는 방법으로 비교 대상이 되는 데이터 주변에 가장 가까이 존재하는 k개의 데이터와 비교해서 판별한다.

## 붓꽃 데이터를 사용해서 붓꽃 종류를 분류하는 모델을 생성하고 학습시킨다.

사이킷런 라이브러리가 제공하는 붓꽃 데이터를 불러온다.

In [3]:
raw_data = datasets.load_iris()
raw_data

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

피처, 레이블 데이터를 저장한다.

In [4]:
xData = raw_data.data # 피처
yData = raw_data.target # 레이블
print(xData.shape, yData.shape)

(150, 4) (150,)


피처, 레이블 데이터를 학습 데이터와 테스트 데이터로 나눈다.

학습 데이터와 테스트 데이터로 나누기 위해서 train_test_split을 import 한다.

In [5]:
from sklearn.model_selection import train_test_split

train_test_split() 함수로 피처와 레이블을 넘겨서 학습 데이터와 테스트 데이터로 나눈다.  
`train_size`로 학습 데이터로 사용할 데이터의 비율을 지정한다.  
`test_size`로 테스트 데이터로 사용할 데이터의 비율을 지정한다.  
train_size, test_size를 모두 생략하면 학습 데이터와 테스트 데이터를 `75:25` 비율로 분할한다.  
`random_state`를 지정하면 train_test_split() 함수를 실행할 때 매번 같은 데이터를 얻을 수 있다. => 항상 같은 결과가 나온다.

`x_train`: 학습 데이터의 피처, `x_test`: 테스트 데이터의 피처, `y_train`: 학습 데이터의 레이블, `y_test`: 테스트 데이터의 레이블

In [6]:
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(112, 4) (38, 4) (112,) (38,)


필요하다면 데이터 스케일링 작업을 한다.  
스케일러는 학습 데이터를 기반으로 실행해야 하기 때문에 `학습 데이터는 학습 후 적용`하고 테스트 `데이터는 학습 데이터로 학습된 스케일러에 적용`만 시킨다.

In [7]:
print(x_train[0])

[5.9 3.  4.2 1.5]


In [8]:
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.

# 학습 데이터는 학습 후 적용한다.
# scaler.fit(x_train) # 학습
# new_scaler = scaler.transform(x_train) # 적용
x_train = scaler.fit_transform(x_train) # 학습 후 적용
print(x_train[0])

# 테스트 데이터는 학습 데이터로 학습된 스케일러를 적용만 한다.
x_test = scaler.transform(x_test)
print(x_test[0])

[ 0.01543995 -0.11925475  0.22512685  0.35579762]
[-0.09984503 -0.57982483  0.72717965  1.51271377]


k최근접 이웃 모델을 만들고 학습시킨다.

k최근접 이웃 모델 알고리즘을 사용하기 위해 KNeighborsClassifier를 import 한다.

In [9]:
from sklearn.neighbors import KNeighborsClassifier

KNeighborsClassifier 클래스로 최근접 이웃의 개수를 넘겨서 k최근접 이웃 모델을 만든다.

In [10]:
model = KNeighborsClassifier(n_neighbors=5)

k최근접 이웃 모델을 학습시키기 위해서 fit() 메소드에 `학습 데이터(x_train)`와 `학습 데이터에 따른 레이블(y_train)`을 순서대로 넘긴다.

In [11]:
model.fit(x_train, y_train)

KNeighborsClassifier()

## 학습된 모델로 테스트 데이터를 예측한다.

predict() 메소드에 `테스트 데이터(x_test)`를 넘겨서 결과를 예측한다. `예측값`을 계산한다.

In [12]:
predict = model.predict(x_test)
print(predict)

[2 1 0 2 0 2 0 1 1 1 2 1 1 1 1 0 1 1 0 0 2 1 0 0 2 0 0 1 1 0 2 1 0 2 2 1 0
 2]


# 모델 성능 평가

## 혼동 행렬(confusion matrix, 오차 행렬)

머신러닝의 분류 모델이 예측한 결과가 실제 정답과 비교해서 얼마나 맞고 틀렸는지를 2차원 표 형태로 정리한 성능 평가 지표이다.

혼동 행렬을 계산하기 위해서 confusion_matrixf를 import 한다.

In [13]:
from sklearn.metrics import confusion_matrix

confusion_matrix() 함수로 `테스트 데이터에 따른 레이블(y_test)`과 `예측값(predict)`을 순서대로 넘겨서 혼동 행렬을 계산한다.

In [14]:
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[13  0  0]
 [ 0 15  1]
 [ 0  0  9]]


## 분류 리포트(classification report)

분류 모델의 성능을 한눈에 파악할 수 있도록 주요 지표(정확도, 정밀도, 재현율, f1 점수)를 요약해서 출력한다.

분류 리포트를 출력하기 위해서 classification_report를 import 한다.

In [15]:
from sklearn.metrics import classification_report

classification_report() 함수로 `테스트 데이터에 따른 레이블(y_test)`과 `예측값(predict)`을 순서대로 넘겨서 분류 리포트를 출력한다.

In [16]:
classification = classification_report(y_test, predict)
print(classification)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      0.94      0.97        16
           2       0.90      1.00      0.95         9

    accuracy                           0.97        38
   macro avg       0.97      0.98      0.97        38
weighted avg       0.98      0.97      0.97        38



In [17]:
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        13
  versicolor       1.00      0.94      0.97        16
   virginica       0.90      1.00      0.95         9

    accuracy                           0.97        38
   macro avg       0.97      0.98      0.97        38
weighted avg       0.98      0.97      0.97        38



## 정확도(accuracy)

머신러닝 분류 모델의 성능을 평가하는 가장 직관적이고 널리쓰이는 지표로, 전체 예측 데이터 중에서 모델이 올바르게 맞춘 비율을 뜻한다.

정확도를 계산하기 위해서 accuracy_score를 import 한다.

In [18]:
from sklearn.metrics import accuracy_score

accuracy_score() 함수로 `테스트 데이터에 따른 레이블(y_test)`과 `예측값(predict)`을 순서대로 넘겨서 정확도를 계산한다.

In [19]:
accuracy = accuracy_score(y_test, predict)
print(accuracy)

0.9736842105263158


## 정밀도(precision) - 혼동 행렬의 세로 방향으로 계산한다.

머신러닝 분류 모델에서 모델이 `positive(양성)라로 예측한 것 중에서 실제로 positive인 것의 비율`을 나타내는 지표를 뜻한다.

정밀도를 계산하기 위해서 precision_score를 import 한다.

In [20]:
from sklearn.metrics import precision_score

precision_score() 함수로 `테스트 데이터에 따른 레이블(y_test)`과 `예측값(predict)`을 순서대로 넘겨서 정밀도를 계산한다.

`average`의 기본값은 binary이고 binary는 레이블의 종류가 딱 2개일 경우를 의미한다. 레이블의 종류가 3개 이상이면 multiclass로 취급된다.  

multiclass일 경우는 binary 대신 None, micro, macro, weighted 중에서 하나를 선택해서 지정해야 한다.  
None: 모든 클래스 각각의 정밀도 값을 그대로 얻어온다. 각 클래스별 성능을 낱낱이 파악 가능하다.  
micro: 마이크로 평균을 얻어온다. 모든 샘플을 개별 단위로 동등하게 취급해서 정확도와 같은 값이 된다.  
macro: 매크로 평균을 얻어온다. 각 클래스별 정밀도를 계산한 뒤, 이 값들의 산술 평균을 계산한다. 모든 클래스를 동등하게 취급한다.  
weighted: 가중 평균을 얻어온다. 각 클래스의 정밀도를 계산한 뒤, 실제 데이터 수(support)의 비중을 가중치로 곱해서 합산한다. 실제 데이터 분포 비중을 반영한다.

In [22]:
precision = precision_score(y_test, predict, average=None)
print(precision)

[1.  1.  0.9]


In [24]:
precision = precision_score(y_test, predict, average='micro')
print(precision)

0.9736842105263158


In [25]:
precision = precision_score(y_test, predict, average='macro')
print(precision)

0.9666666666666667


In [26]:
precision = precision_score(y_test, predict, average='weighted')
print(precision)

0.9763157894736842


## 재현율(recall) - 혼동 행렬의 가로 방향으로 계산한다.

머신러닝 분류 모델에서 모델이 `positive(양성)인 데이터 중에서 positive라고 예측한 비율`을 나타내는 지표를 뜻한다.

재현율를 계산하기 위해서 recall_score import 한다.

In [27]:
from sklearn.metrics import recall_score

recall_score() 함수로 테스트 데이터에 따른 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.

`average`는 정밀도와 사용 방법이 동일하다.

In [28]:
recall = recall_score(y_test, predict, average=None)
print(recall)

[1.     0.9375 1.    ]


# f1 점수(f1-score)

정밀도와 재현율의 `조화 평균`으로 계산한 지표를 뜻한다.

f1 점수를 계산하기 위해서 f1_score import 한다.

In [29]:
from sklearn.metrics import f1_score

f1_score 함수로 테스트 데이터에 따른 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 점수를 계산한다.

`average`는 정밀도, 재현율과 사용 방법이 동일하다.

In [30]:
f1 = f1_score(y_test, predict, average=None)
print(f1)

[1.         0.96774194 0.94736842]
